# 3-6. GraphRAG: Neo4j + Microsoft GraphRAG 개념

⏱ **소요시간**: 45분

## 학습 목표
- `langchain-neo4j`의 `Neo4jGraph`로 Neo4j에 연결하고 Cypher로 그래프를 구축할 수 있다.
- `LLMGraphTransformer`로 비정형 한국어 텍스트에서 **자동 지식그래프**를 추출할 수 있다.
- `GraphCypherQAChain`으로 자연어 질의를 Cypher로 변환해 그래프 기반 QA를 수행할 수 있다.
- Microsoft GraphRAG의 **community detection · local vs global search** 개념을 설명할 수 있다.
- Vector RAG와 GraphRAG를 **multi-hop 질문**에서 비교할 수 있다.

## 핵심 키워드
Neo4j 🔒 · Cypher · LLMGraphTransformer · GraphCypherQAChain · Community Detection · Leiden · Local vs Global Search · Multi-Hop

## 사전 준비
- `docker compose up -d neo4j` 실행 필요 (repo 루트의 `docker-compose.yml`)
- 접속 정보: `bolt://localhost:7687` / user=`neo4j` / password=`password`


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_llm, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. Neo4j 연결 테스트 🔒

In [ ]:
# 주의: docker compose up -d neo4j 가 먼저 실행되어 있어야 한다.
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url='bolt://localhost:7687',
    username='neo4j',
    password='password',
)

# 기존 그래프 초기화 (교육용)
graph.query('MATCH (n) DETACH DELETE n')
print('Neo4j 연결 OK, 기존 노드 삭제됨')


## 2. CSV → Cypher 로 그래프 구축

`data/kg_sample.csv`는 `subject, relation, object` 트리플 형식이다.

In [ ]:
import csv
from pathlib import Path

csv_path = Path('../../data/kg_sample.csv')
with csv_path.open(encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'트리플 수: {len(rows)}')
print('샘플:', rows[:3])


In [ ]:
# MERGE로 중복 없이 노드/관계 생성. 관계 타입은 APOC의 dynamic create 사용.
# APOC이 활성화되어 있어야 한다 (docker-compose.yml에서 이미 설정됨).
cypher = '''
UNWIND $rows AS row
MERGE (s:Entity {name: row.subject})
MERGE (o:Entity {name: row.object})
WITH s, o, row
CALL apoc.create.relationship(s, row.relation, {}, o) YIELD rel
RETURN count(rel) AS created
'''
result = graph.query(cypher, params={'rows': rows})
print('생성된 관계 수:', result)
graph.refresh_schema()
print('\n[Neo4j 스키마]')
print(graph.schema[:500])


## 3. Cypher 직접 질의로 multi-hop 확인

In [ ]:
# "삼성전자의 자회사와 그 자회사의 사업분야는?"
q = '''
MATCH (s:Entity {name:'삼성전자'})-[:자회사]->(sub:Entity)-[:사업분야]->(dom:Entity)
RETURN sub.name AS 자회사, dom.name AS 사업분야
'''
print(graph.query(q))


## 4. 자연어 → Cypher 자동 생성: `GraphCypherQAChain`

사용자가 한국어로 물으면 LLM이 스키마를 참고해 Cypher를 작성, 실행 결과를 자연어로 답변한다.

In [ ]:
from langchain_neo4j import GraphCypherQAChain

llm = get_llm(temperature=0.0)
qa_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,  # 교육용: 프로덕션에서는 read-only 사용자로 제한 권장
)

ans = qa_chain.invoke({'query': '삼성전자의 자회사는 어디이고, 그 자회사는 어떤 사업을 하는가?'})
print('\n[최종 답변]\n', ans['result'])


## 5. `LLMGraphTransformer`로 비정형 텍스트 → 지식그래프

`data/corpus_ko.txt`를 LLM에게 읽혀 자동으로 엔터티·관계를 뽑아낸다.

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

docs = load_any('../../data/corpus_ko.txt')
transformer = LLMGraphTransformer(llm=llm)
graph_docs = transformer.convert_to_graph_documents(docs)

print(f'추출된 그래프 문서: {len(graph_docs)}')
for gd in graph_docs[:1]:
    print('\n[Nodes]', gd.nodes[:5])
    print('[Relationships]', gd.relationships[:5])


In [ ]:
# Neo4j에 병합 (`baseEntityLabel=True`로 Entity 공통 라벨 부여)
graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=False)
graph.refresh_schema()
print('자동 추출 그래프 삽입 완료')
print(graph.schema[:500])


## 6. Microsoft GraphRAG 개념 (실행 없음, 개념만)

[microsoft/graphrag](https://github.com/microsoft/graphrag)는 2024년 공개된 엔드투엔드 파이프라인이다.

### 6.1 파이프라인
```
원본 문서 ─▶ 엔터티/관계 추출 (LLM)
           ─▶ 지식그래프 구축
           ─▶ Community Detection (Leiden 알고리즘)
           ─▶ 각 community의 요약을 LLM으로 생성 (Community Report)
           ─▶ 쿼리 시: Local Search | Global Search
```

### 6.2 Community Detection (Leiden)
- 그래프를 "밀도 높은 부분집합"으로 분할. 계층적으로 여러 해상도의 community 생성.
- 예: 전자금융 약관 그래프에서 "전산 보안 관련 노드들"이 하나의 community로 묶임.
- 각 community에 대해 LLM이 "이 커뮤니티는 ~에 관한 내용이다"라는 **요약 리포트**를 작성.

### 6.3 Local vs Global Search

| | Local Search | Global Search |
|---|-------------|---------------|
| 질문 예시 | "삼성전자 자회사는?" (특정 엔터티) | "이 코퍼스 전반의 주요 주제는?" (전체 요약) |
| 동작 | 쿼리 관련 노드 근방 이웃 + 해당 local community 리포트 | 모든 community 리포트를 map-reduce로 합성 |
| 강점 | 정밀 사실 질의 | 전체 테마·추세 질의 |

> **핵심 통찰**: 기존 Vector RAG는 "질문과 유사한 청크 top-K"만 보기 때문에 **전체를 관통하는 질의**("이 책에서 주로 다루는 테마 3가지는?")에 약하다. GraphRAG는 community 리포트 덕분에 이런 질의에 자연스럽게 답한다.

### 6.4 왜 이 과정에서 GraphRAG 코드를 실행하지 않는가?
- `graphrag` 공식 CLI는 OpenAI 호출이 수백~수천 회 필요하다 (≈ $20~$100).
- 교육 환경에서는 `langchain-neo4j` + `LLMGraphTransformer` 조합으로 충분히 핵심 개념을 체험할 수 있다.
- 실제 도입 시에는 자사 코퍼스에 대해 1회 비용을 들여 인덱스를 구축하고 재사용한다.


## 7. Vector RAG vs GraphRAG: Multi-Hop 비교

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# CSV 트리플을 자연어 문장으로 변환해 Vector RAG 코퍼스 구성
triple_texts = [f'{r["subject"]}의 {r["relation"]}은(는) {r["object"]}이다.' for r in rows]
from langchain_core.documents import Document
triple_docs = [Document(page_content=t) for t in triple_texts]
vdb = FAISS.from_documents(triple_docs, get_embeddings())
vretriever = vdb.as_retriever(search_kwargs={'k': 4})

rag_prompt = ChatPromptTemplate.from_template(
    '다음 문장들만 근거로 사용해 질문에 답하라.\n\n{ctx}\n\n질문: {q}\n답변:'
)
rag_chain = rag_prompt | llm | StrOutputParser()

multi_hop_q = '삼성전자의 자회사는 어디이고, 그 자회사의 사업분야는 무엇인가?'

ctx = '\n'.join(d.page_content for d in vretriever.invoke(multi_hop_q))
print('[Vector RAG 컨텍스트]')
print(ctx)
print('\n[Vector RAG 답변]')
print(rag_chain.invoke({'ctx': ctx, 'q': multi_hop_q}))


In [ ]:
print('[GraphRAG 답변]')
print(qa_chain.invoke({'query': multi_hop_q})['result'])


### 관찰 포인트
- Vector RAG는 "삼성전자 자회사 삼성SDI" 문장과 "삼성SDI 사업분야 이차전지" 문장을 **모두** top-K에 담아야 정답이 나온다. top-K가 작으면 누락 발생.
- GraphRAG는 Cypher로 `(삼성전자)-[:자회사]->(?)-[:사업분야]->(?)` **정확히 2-hop 경로**를 탐색하므로 top-K 의존이 없다.
- 다만 GraphRAG는 그래프 구축·스키마 관리 비용이 크다. 질의 유형이 **엔터티 간 관계 중심**이 아니라면 Vector RAG로 충분하다.

### 선택 가이드
| 코퍼스·질의 특성 | 추천 |
|-----------------|------|
| 비정형 서술 위주, 의미 기반 질의 | Vector RAG |
| 관계/계층 多, multi-hop 질의 多 | GraphRAG |
| 전체 코퍼스 요약·주제 질의 | MS GraphRAG (Global Search) |
| 둘 다 필요 | Hybrid: 엔터티는 그래프, 서술은 벡터 |
